# 🤖 Tu primera llamada a un LLM vía API

**Objetivo:** Realizar tu primera integración real con un LLM desde código Python: enviar un mensaje, recibir una respuesta y entender cómo cambia el comportamiento del modelo.

---

Este notebook cubre **ambos proveedores** (Anthropic y OpenAI). Ejecuta solo las secciones del proveedor que hayas configurado.

| Nivel | Descripción | Estado |
|-------|-------------|--------|
| Nivel 1 | Llamada básica a la API | Obligatorio |
| Nivel 2 | System prompt con rol específico | Obligatorio |
| Nivel 3 | Metadatos y cálculo de coste | Opcional |

---

## ⚙️ 0. Configuración inicial

Instala las librerías necesarias y configura tu API key.

In [ ]:
# Instalar las librerías de ambos proveedores
!pip install anthropic openai --quiet

In [ ]:
# ──────────────────────────────────────────────────────────────────
# OPCIÓN A: Cargar la API key desde Google Colab Secrets (recomendado)
# ──────────────────────────────────────────────────────────────────
# 1. Haz clic en el icono 🔑 (Secrets) en el panel izquierdo de Colab
# 2. Añade una entrada llamada ANTHROPIC_API_KEY  y/o  OPENAI_API_KEY
# 3. Activa el acceso al notebook
# 4. Ejecuta esta celda

from google.colab import userdata
import os

try:
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    print("✅ ANTHROPIC_API_KEY cargada correctamente")
except Exception:
    ANTHROPIC_API_KEY = None
    print("⚠️  ANTHROPIC_API_KEY no encontrada en Secrets")

try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    print("✅ OPENAI_API_KEY cargada correctamente")
except Exception:
    OPENAI_API_KEY = None
    print("⚠️  OPENAI_API_KEY no encontrada en Secrets")

In [ ]:
# ──────────────────────────────────────────────────────────────────
# OPCIÓN B: Introducir la API key manualmente (menos seguro)
# Solo si la Opción A no funcionó. Nunca compartas este notebook con
# la key visible.
# ──────────────────────────────────────────────────────────────────

# Descomenta y rellena SOLO el proveedor que vayas a usar:

# ANTHROPIC_API_KEY = "sk-ant-..."   # Tu clave de Anthropic
# OPENAI_API_KEY    = "sk-..."       # Tu clave de OpenAI

---

# 🟦 PROVEEDOR: ANTHROPIC (Claude)

Ejecuta esta sección si configuraste una cuenta en **https://console.anthropic.com**

## 📗 Nivel 1 — Llamada básica (Anthropic)

**Objetivo:** Enviar un mensaje simple y recibir una respuesta.

Si ves una respuesta sin errores, ¡el setup es correcto! ✅

In [ ]:
import anthropic

# Inicializar el cliente
client_anthropic = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# Mensaje que enviaremos al modelo
mensaje = "¿Cuánto tiempo se tarda en desarrollar una aplicación web sencilla?"

# ── Llamada a la API ────────────────────────────────────────────────
respuesta = client_anthropic.messages.create(
    model="claude-haiku-4-5-20251001",   # Modelo más económico para practicar
    max_tokens=512,
    messages=[
        {"role": "user", "content": mensaje}
    ]
)

# Mostrar la respuesta
print("📨 Mensaje enviado:")
print(f"  {mensaje}")
print()
print("🤖 Respuesta del modelo:")
print("-" * 60)
print(respuesta.content[0].text)

## 📘 Nivel 2 — System prompt con rol (Anthropic)

**Objetivo:** Añadir un system prompt que defina un rol y comparar la respuesta con la del Nivel 1.

**Observa:**
- El **tono** (más técnico/formal vs. genérico)
- El **nivel de detalle** (más específico, más descomposición de tareas)
- La **estructura** (¿usa fases, estimaciones numéricas, rangos?)

In [ ]:
# System prompt que define el rol del asistente
system_prompt = """
Eres un experto en estimación de proyectos de software con más de 15 años de experiencia.
Trabajas como consultor para startups y empresas medianas.
Cuando te pregunten sobre tiempos o esfuerzo, siempre:
1. Desglosas el trabajo en fases (análisis, diseño, desarrollo, pruebas, despliegue).
2. Das rangos de tiempo realistas (mínimo – máximo) en función del tamaño del equipo.
3. Señalas los factores de riesgo más comunes que pueden afectar la estimación.
Respondes de forma concisa y estructurada, usando listas cuando sea útil.
"""

# Mismo mensaje que en el Nivel 1
mensaje = "¿Cuánto tiempo se tarda en desarrollar una aplicación web sencilla?"

# ── Llamada a la API con system prompt ─────────────────────────────
respuesta_con_rol = client_anthropic.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=512,
    system=system_prompt,              # <── aquí añadimos el system prompt
    messages=[
        {"role": "user", "content": mensaje}
    ]
)

print("📨 Mensaje enviado (idéntico al Nivel 1):")
print(f"  {mensaje}")
print()
print("🤖 Respuesta CON system prompt (rol de experto):")
print("-" * 60)
print(respuesta_con_rol.content[0].text)

In [ ]:
# ── Comparación lado a lado ─────────────────────────────────────────
print("=" * 60)
print("COMPARACIÓN: SIN rol  vs.  CON rol")
print("=" * 60)
print()
print("❌ Sin system prompt:")
print("-" * 60)
print(respuesta.content[0].text)
print()
print("✅ Con system prompt (experto en estimación):")
print("-" * 60)
print(respuesta_con_rol.content[0].text)

## 📙 Nivel 3 — Metadatos y coste estimado (Anthropic) [Opcional]

**Objetivo:** Extraer la información de uso de la respuesta y calcular el coste.

**Precios de referencia (mayo 2026) — por millón de tokens:**

| Modelo | Input | Output |
|--------|-------|--------|
| claude-haiku-4-5 | $1.00 | $5.00 |
| claude-sonnet-4-6 | $3.00 | $15.00 |
| claude-opus-4-6 | $5.00 | $25.00 |

Fuente: [anthropic.com/pricing](https://www.anthropic.com/pricing)

In [ ]:
# Precios por millón de tokens (actualiza si cambian)
PRECIOS = {
    "claude-haiku-4-5-20251001": {"input": 1.00,  "output": 5.00},
    "claude-sonnet-4-6":         {"input": 3.00,  "output": 15.00},
    "claude-opus-4-6":           {"input": 5.00,  "output": 25.00},
}

def analizar_uso_anthropic(respuesta, etiqueta="Llamada"):
    """
    Extrae metadatos de una respuesta de la API de Anthropic
    y calcula el coste estimado.
    """
    uso   = respuesta.usage
    modelo = respuesta.model

    tokens_entrada = uso.input_tokens
    tokens_salida  = uso.output_tokens
    total_tokens   = tokens_entrada + tokens_salida

    # Calcular coste
    precios_modelo = PRECIOS.get(modelo, {"input": 0, "output": 0})
    coste_entrada  = (tokens_entrada / 1_000_000) * precios_modelo["input"]
    coste_salida   = (tokens_salida  / 1_000_000) * precios_modelo["output"]
    coste_total    = coste_entrada + coste_salida

    print(f"📊 {etiqueta}")
    print("-" * 45)
    print(f"  Modelo utilizado : {modelo}")
    print(f"  Tokens de entrada: {tokens_entrada:,}")
    print(f"  Tokens de salida : {tokens_salida:,}")
    print(f"  Total tokens     : {total_tokens:,}")
    print(f"  Stop reason      : {respuesta.stop_reason}")
    print()
    print(f"  💰 Coste entrada : ${coste_entrada:.6f}")
    print(f"  💰 Coste salida  : ${coste_salida:.6f}")
    print(f"  💰 COSTE TOTAL   : ${coste_total:.6f}")

    if precios_modelo["input"] == 0:
        print(f"  ⚠️  Modelo '{modelo}' no encontrado en tabla de precios.")
        print("     Actualiza el diccionario PRECIOS con el precio correcto.")
    print()
    return coste_total

# Analizar las dos llamadas
coste_nivel1 = analizar_uso_anthropic(respuesta,          "Nivel 1 — Sin system prompt")
coste_nivel2 = analizar_uso_anthropic(respuesta_con_rol,  "Nivel 2 — Con system prompt")

print("=" * 45)
print(f"  💰 Coste combinado de ambas llamadas: ${coste_nivel1 + coste_nivel2:.6f}")
print("=" * 45)

---

# 🟩 PROVEEDOR: OPENAI (GPT)

Ejecuta esta sección si configuraste una cuenta en **https://platform.openai.com**

## 📗 Nivel 1 — Llamada básica (OpenAI)

**Objetivo:** Enviar un mensaje simple y recibir una respuesta.

Si ves una respuesta sin errores, ¡el setup es correcto! ✅

In [ ]:
from openai import OpenAI

# Inicializar el cliente
client_openai = OpenAI(api_key=OPENAI_API_KEY)

# Mensaje que enviaremos al modelo
mensaje = "¿Cuánto tiempo se tarda en desarrollar una aplicación web sencilla?"

# ── Llamada a la API ────────────────────────────────────────────────
respuesta_oai = client_openai.chat.completions.create(
    model="gpt-4o-mini",   # Modelo más económico para practicar
    max_tokens=512,
    messages=[
        {"role": "user", "content": mensaje}
    ]
)

# Mostrar la respuesta
print("📨 Mensaje enviado:")
print(f"  {mensaje}")
print()
print("🤖 Respuesta del modelo:")
print("-" * 60)
print(respuesta_oai.choices[0].message.content)

## 📘 Nivel 2 — System prompt con rol (OpenAI)

**Objetivo:** Añadir un system prompt que defina un rol y comparar la respuesta con la del Nivel 1.

**Observa:**
- El **tono** (más técnico/formal vs. genérico)
- El **nivel de detalle** (más específico, más descomposición de tareas)
- La **estructura** (¿usa fases, estimaciones numéricas, rangos?)

In [ ]:
# System prompt que define el rol del asistente
system_prompt_oai = """
Eres un experto en estimación de proyectos de software con más de 15 años de experiencia.
Trabajas como consultor para startups y empresas medianas.
Cuando te pregunten sobre tiempos o esfuerzo, siempre:
1. Desglosas el trabajo en fases (análisis, diseño, desarrollo, pruebas, despliegue).
2. Das rangos de tiempo realistas (mínimo – máximo) en función del tamaño del equipo.
3. Señalas los factores de riesgo más comunes que pueden afectar la estimación.
Respondes de forma concisa y estructurada, usando listas cuando sea útil.
"""

# Mismo mensaje que en el Nivel 1
mensaje = "¿Cuánto tiempo se tarda en desarrollar una aplicación web sencilla?"

# ── Llamada a la API con system prompt ─────────────────────────────
respuesta_oai_con_rol = client_openai.chat.completions.create(
    model="gpt-4o-mini",
    max_tokens=512,
    messages=[
        {"role": "system", "content": system_prompt_oai},  # <── system prompt
        {"role": "user",   "content": mensaje}
    ]
)

print("📨 Mensaje enviado (idéntico al Nivel 1):")
print(f"  {mensaje}")
print()
print("🤖 Respuesta CON system prompt (rol de experto):")
print("-" * 60)
print(respuesta_oai_con_rol.choices[0].message.content)

In [ ]:
# ── Comparación lado a lado ─────────────────────────────────────────
print("=" * 60)
print("COMPARACIÓN: SIN rol  vs.  CON rol")
print("=" * 60)
print()
print("❌ Sin system prompt:")
print("-" * 60)
print(respuesta_oai.choices[0].message.content)
print()
print("✅ Con system prompt (experto en estimación):")
print("-" * 60)
print(respuesta_oai_con_rol.choices[0].message.content)

## 📙 Nivel 3 — Metadatos y coste estimado (OpenAI) [Opcional]

**Objetivo:** Extraer la información de uso de la respuesta y calcular el coste.

**Precios de referencia (mayo 2026) — por millón de tokens:**

| Modelo | Input | Output |
|--------|-------|--------|
| gpt-4o-mini | $0.15 | $0.60 |
| gpt-4o | $2.50 | $10.00 |
| o1-mini | $1.10 | $4.40 |

Fuente: [platform.openai.com/pricing](https://platform.openai.com/pricing)

In [ ]:
# Precios por millón de tokens (actualiza si cambian)
PRECIOS_OAI = {
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "o1-mini":     {"input": 1.10,  "output": 4.40},
}

def analizar_uso_openai(respuesta, etiqueta="Llamada"):
    """
    Extrae metadatos de una respuesta de la API de OpenAI
    y calcula el coste estimado.
    """
    uso    = respuesta.usage
    modelo = respuesta.model

    tokens_entrada = uso.prompt_tokens
    tokens_salida  = uso.completion_tokens
    total_tokens   = uso.total_tokens

    # Calcular coste
    # El modelo devuelto puede incluir una versión (ej: "gpt-4o-mini-2024-07-18")
    # buscamos la clave más corta que coincida
    clave_modelo = next(
        (k for k in PRECIOS_OAI if modelo.startswith(k)),
        None
    )
    precios_modelo = PRECIOS_OAI.get(clave_modelo, {"input": 0, "output": 0})

    coste_entrada  = (tokens_entrada / 1_000_000) * precios_modelo["input"]
    coste_salida   = (tokens_salida  / 1_000_000) * precios_modelo["output"]
    coste_total    = coste_entrada + coste_salida

    finish_reason = respuesta.choices[0].finish_reason

    print(f"📊 {etiqueta}")
    print("-" * 45)
    print(f"  Modelo utilizado : {modelo}")
    print(f"  Tokens de entrada: {tokens_entrada:,}")
    print(f"  Tokens de salida : {tokens_salida:,}")
    print(f"  Total tokens     : {total_tokens:,}")
    print(f"  Finish reason    : {finish_reason}")
    print()
    print(f"  💰 Coste entrada : ${coste_entrada:.6f}")
    print(f"  💰 Coste salida  : ${coste_salida:.6f}")
    print(f"  💰 COSTE TOTAL   : ${coste_total:.6f}")

    if precios_modelo["input"] == 0:
        print(f"  ⚠️  Modelo '{modelo}' no encontrado en tabla de precios.")
        print("     Actualiza el diccionario PRECIOS_OAI con el precio correcto.")
    print()
    return coste_total

# Analizar las dos llamadas
coste_n1_oai = analizar_uso_openai(respuesta_oai,          "Nivel 1 — Sin system prompt")
coste_n2_oai = analizar_uso_openai(respuesta_oai_con_rol,  "Nivel 2 — Con system prompt")

print("=" * 45)
print(f"  💰 Coste combinado de ambas llamadas: ${coste_n1_oai + coste_n2_oai:.6f}")
print("=" * 45)

---

## 🧠 Reflexión final

Usa esta celda para anotar tus observaciones después de ejecutar el notebook.

### Comparativa de respuestas

| Aspecto | Sin system prompt | Con system prompt |
|---------|:-----------------:|:-----------------:|
| Tono | genérico | técnico/experto |
| Nivel de detalle | básico | estructurado |
| Uso de fases | ❌ / ✅ | ❌ / ✅ |
| Rangos de tiempo | ❌ / ✅ | ❌ / ✅ |
| Factores de riesgo | ❌ / ✅ | ❌ / ✅ |

*(Rellena la tabla con tus observaciones reales)*

### Preguntas para reflexionar

1. ¿Qué diferencias notaste entre la respuesta con y sin system prompt?
2. ¿Qué ocurre si cambias el rol en el system prompt (ej: "eres un desarrollador junior" vs. "eres un CTO")?
3. ¿Para qué tipo de aplicaciones justificaría usar un modelo más caro (Opus/GPT-4o)?
4. ¿Qué estrategias reducirían el coste en producción? (pista: mira batch API y prompt caching)

---

## 🔗 Recursos adicionales

- **Anthropic Quickstart:** https://docs.anthropic.com/en/docs/quickstart
- **Anthropic Pricing:** https://www.anthropic.com/pricing
- **OpenAI API Reference:** https://platform.openai.com/docs/api-reference
- **OpenAI Pricing:** https://platform.openai.com/pricing
- **Prompt Engineering Guide (Anthropic):** https://docs.claude.com/en/docs/build-with-claude/prompt-engineering/overview